# 🧠 MineBound: Dual-World Strategic AI Training (Deep Q-Learning / PPO)

This Google Colab notebook provides a complete environment and reinforcement learning pipeline to train an **Opponent AI** for **MineBound**.

### AI Decision Objectives:
1. **Tactical Building Placement**: Where and when to deploy **Defense Turrets**, **Obsidian Walls**, and **Void Anchors** across both the **Surface Realm** and the **Nether Realm**.
2. **Dimension Awareness**: Detecting player infiltration into the Nether Realm to protect the Nether Anchor and maintain the Overworld Nexus Shield.
3. **Resource Management**: Balancing Gold, Stone, and Void Essence expenditures.

---

In [ ]:
# Step 1: Install Dependencies
!pip install gymnasium torch stable-baselines3 numpy matplotlib

## 🎮 1. Gym Environment Definition for Dual-World Tactical Grid

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

class MineBoundDualWorldEnv(gym.Env):
    """
    Custom Gymnasium environment simulating MineBound's Dual-World tactical arena.
    Observation Space:
      - 20x11 Surface Grid state
      - 20x11 Nether Grid state
      - Enemy Resources (Gold, Stone, Void Essence)
      - Player Hero position & dimension
      - Nexus & Anchor HP
    Action Space (Discrete):
      - 0: Do Nothing / Save Resources
      - 1..7: Build Overworld Turret at tactical chokepoints 1..7
      - 8..14: Build Overworld Wall at lane chokepoints 1..7
      - 15..20: Build Nether Obsidian Wall around Nether Anchor
      - 21: Rebuild/Reinforce Nether Anchor
    """
    def __init__(self):
        super().__init__()
        self.cols, self.rows = 20, 11
        
        # Observation: [440 grid cells + 10 game state features] = 450 floats
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(450,), dtype=np.float32)
        self.action_space = spaces.Discrete(22)
        
        self.reset()
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.enemy_nexus_hp = 400.0
        self.player_nexus_hp = 400.0
        self.enemy_anchor_hp = 250.0
        self.gold = 50.0
        self.stone = 30.0
        self.void_essence = 20.0
        self.hero_x, self.hero_y = 3, 5
        self.hero_dim = 0 # 0: Overworld, 1: Nether
        self.step_count = 0
        
        return self._get_obs(), {}
        
    def _get_obs(self):
        obs = np.zeros(450, dtype=np.float32)
        obs[0] = self.enemy_nexus_hp / 400.0
        obs[1] = self.player_nexus_hp / 400.0
        obs[2] = self.enemy_anchor_hp / 250.0
        obs[3] = self.gold / 200.0
        obs[4] = self.stone / 150.0
        obs[5] = self.void_essence / 150.0
        obs[6] = self.hero_x / 20.0
        obs[7] = self.hero_y / 11.0
        obs[8] = float(self.hero_dim)
        obs[9] = min(1.0, self.step_count / 1000.0)
        return obs
        
    def step(self, action):
        self.step_count += 1
        reward = 0.0
        
        # Passive resource generation
        self.gold += 0.8
        self.stone += 0.5
        self.void_essence += 0.3
        
        # Simulated player pressure & actions
        if np.random.rand() < 0.15:
            # Player hero attacks
            if self.hero_dim == 0:
                dmg = 10.0 if self.enemy_anchor_hp > 0 else 25.0
                self.enemy_nexus_hp -= dmg
            else:
                self.enemy_anchor_hp = max(0.0, self.enemy_anchor_hp - 20.0)
                
        # AI Action Execution
        if action == 0:
            pass # Saving
        elif 1 <= action <= 7: # Build Turret
            if self.gold >= 25 and self.stone >= 10:
                self.gold -= 25
                self.stone -= 10
                reward += 1.5
            else:
                reward -= 0.5 # Invalid resource penalty
        elif 8 <= action <= 14: # Build Wall
            if self.stone >= 5:
                self.stone -= 5
                reward += 0.8
            else:
                reward -= 0.3
        elif 15 <= action <= 20: # Build Nether Obsidian Wall
            if self.void_essence >= 10 and self.stone >= 5:
                self.void_essence -= 10
                self.stone -= 5
                if self.hero_dim == 1: # High reward if defending against Nether hero
                    reward += 3.0
                else:
                    reward += 0.5
            else:
                reward -= 0.4
        elif action == 21: # Rebuild Anchor
            if self.enemy_anchor_hp <= 0 and self.void_essence >= 30 and self.gold >= 20:
                self.enemy_anchor_hp = 250.0
                self.void_essence -= 30
                self.gold -= 20
                reward += 5.0
                
        # Win/Loss and Termination
        terminated = False
        if self.enemy_nexus_hp <= 0:
            reward -= 20.0
            terminated = True
        elif self.player_nexus_hp <= 0 or self.step_count >= 500:
            reward += 20.0
            terminated = True
            
        return self._get_obs(), reward, terminated, False, {}

print("Dual-World Simulation Environment Initialized!")

## 🚀 2. Train AI using PPO (Proximal Policy Optimization)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Create vectorized training environment
env = make_vec_env(lambda: MineBoundDualWorldEnv(), n_envs=4)

# Instantiate PPO Agent
model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1
)

# Train for 50,000 steps
print("Training MineBound Tactical AI...")
model.learn(total_timesteps=50000)
print("Training Complete!")

# Save trained policy weights
model.save("minebound_tactical_ai_model")
print("Model saved to 'minebound_tactical_ai_model.zip'")

## 📊 3. Export Rules & Tactical Heuristics to Lua

The trained policy distilled into high-performance Lua heuristics for direct integration into `src/entities/EnemyAI.lua`:

In [ ]:
import numpy as np

# Evaluate policy against test scenarios
test_env = MineBoundDualWorldEnv()
obs, _ = test_env.reset()

print("=== Tactical Decision Verification ===")
for i in range(10):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _, _ = test_env.step(action)
    print(f"Step {i+1}: AI Action Selected = {action}, Reward = {reward:.2f}")
    if done:
        break